In [6]:
import pandas as pd
from datetime import datetime, timedelta
import sys
import cwms
import numpy as np
import time

### Initialize CDA root and key

In [7]:
office_ids = ['LRL','MVP']
apiRoot_dev = "http://localhost:8081/cwms-data/"
apiKey_dev = "apikey testkey12345677"
api = cwms.api.init_session(api_root=apiRoot_dev, api_key=apiKey_dev)

### Store Locations to Database

In [ ]:
def default_val(value, default):
    if pd.isna(value) or value == 'Unknown or Not Applicable':
        value = default
    return value

def store_multi_location_df(locations):
    for i,row in locations.iterrows():
        if row['active']:
            loc_json = {
              "office-id": row['office'],  # required
              "name": row['name'],    #required
              "latitude": float(default_val(row['latitude'],'38.0')),  #required
              "longitude": float(default_val(row['longitude'],'-85.0')),  #required
              "active": row['active'],  #required
              "public-name": default_val(row['public-name'],row['name']),
              "long-name": row['long-name'],
              "description": row['description'],
              "timezone-name": default_val(row['time-zone'],'US/Eastern'), #required
              "location-type": row['type'], 
              "location-kind": row['kind'],  #required
              "nation": 'US',   #required and abbreviated
              #"state-initial": row['state'],  #Saving state doesn't work.
              #"county-name": row['county'],
              "nearest-city": row['nearest-city'],
              "horizontal-datum": default_val(row['horizontal-datum'],'NAD27'),  #required
              #"published-longitude": float(row['published-longitude']),
              #"published-latitude": float(row['published-latitude']),
              "vertical-datum": row['vertical-datum'],
              "elevation": float(row['elevation']),
              "map-label": row['map-label'],
              "bounding-office-id": row['bounding-office'],
              "elevation-units": row['unit']
            }
            clean_dict = {k: loc_json[k] for k in loc_json if not pd.isna(loc_json[k])}
            #try:
            cwms.store_location(data = clean_dict, fail_if_exists=False)
            #except:
            #    print(clean_dict)
            #    print('save failed')

In [9]:
for office_id in office_ids:
    locations = pd.read_csv(f'data/{office_id}_locations_data.csv')
    store_multi_location_df(locations)

ERROR:root:CDA Error: response=<Response [409]>


ApiError: CWMS API Error (http://localhost:8081/cwms-data/locations?fail-if-exists=True) Conflict. {"message":"Already exists","incidentIdentifier":"8af100ce-92f4-4a02-ab95-6839e593f6c2","source":"Database","details":{"message":"The location with name: Taylorsville-Lake already exists in office: LRL"}}

#### Check if Locations are present

In [ ]:
location_check = cwms.get_locations_catalog(office_id='LRL').df

In [ ]:
location_check

In [ ]:
location_check = cwms.get_locations_catalog(office_id='MVP').df

In [ ]:
location_check

### Store Timeseries Values To Database

In [ ]:
for office_id in office_ids:
    print(f'importing timeseries for {office_id}')
    ts_ids = pd.read_csv(f'data/{office_id}_timeseries_ids_used.csv')
    multi_ts_melt = pd.read_parquet(f'data/{office_id}_timeseries_values_melted.parquet')
    multi_ts_melt = multi_ts_melt.dropna(subset=['value'])
    multi_ts_melt_used = multi_ts_melt[multi_ts_melt['ts_id'].isin(ts_ids['ts_id'])]
    cwms.store_multi_timeseries_df(data=multi_ts_melt_used,office_id=office_id,max_workers=10)

#### Check Timeseries Values

In [ ]:
ts_ids = cwms.get_timeseries_catalog(office_id='MVP',timeseries_group_like=None,page_size=10000,include_extents=True).df

In [ ]:
ts_ids

In [ ]:
start_date = pd.to_datetime('03/01/2025').tz_localize('UTC')
end_date = pd.to_datetime('06/13/2025').tz_localize('UTC')
data = cwms.get_timeseries(ts_id='ZUMM5.Stage.Inst.~15Minutes.0.rev-USGS',office_id='MVP',begin=start_date,end=end_date)

In [ ]:
data.df